# fMRI Reliability Pipeline
- Designed to parametrically analyze fMRI files to identify which pipelines produce the most robust results.

In [ ]:
import os
import pandas as pd
import itertools
import numpy as np

from scipy.spatial.distance import pdist
from scipy.stats import ttest_rel
from statsmodels.stats.anova import AnovaRM

# Plotting Packages 
import seaborn as sns
import matplotlib.pyplot as plt
import ptitprince as pt # Only needed for raincloud plot
import configFunctions as config
from nilearn import plotting

# Variables
dataFile = 'coordinatesAll.csv' # Path to the CSV file containing the coordinates data

# Toggle nilearn plotting
plot_nilearn = False

# Output directory
output_dir = 'output'
os.makedirs(output_dir, exist_ok=True)

### Required files
- 'coordinatesAll.csv' - generated by serverFunctions, final output of mergeOutputs.sh.

In [ ]:
# A configuration function to make output images easier to modify for publication.
config.svg_editing()

In [ ]:
# load data, inspect
df = pd.read_csv(dataFile)
df.head()

### Intermediate clean up

In [ ]:
# Pull out and show the rows which include any nans
rows_with_nan = df[df.isna().any(axis=1)]
rows_with_nan.to_csv(os.path.join(output_dir, 'coordinatesAll_NaNRows.csv'), index=False)
subj_with_nan = rows_with_nan['subject'].unique()
print(f'Subjects with NaNs - {len(subj_with_nan)}')
print(f'List: {subj_with_nan}')
print(f'Dataframe filtered to remove subjects with any NaNs')

df = df[~df['subject'].isin(subj_with_nan)]
# ['107','103','99','45','83','108','110','88','109','115','111','15','116','112','104','100','117','105','101','118','102']
print(f'{len(df['subject'].unique())} Subjects Remaining...')

In [ ]:
# Group by subject and condition
grouping_vars = ['subject', 'target', 'clustMask', 'sequence_type']

subjVec = df['subject'].unique()
targetVec = df['target'].unique()
cMaskVec = df['clustMask'].unique()
seqTypeVec = df['sequence_type'].unique()

# Create an array with each possible combo.
combinations = list(itertools.product(subjVec, targetVec, cMaskVec, seqTypeVec))

# Put into pandas dataframe
combodf = pd.DataFrame(combinations, columns=['subject', 'target', 'clustMask', 'sequence_type'])
combodf.head()

In [ ]:
# Perform the analysis - span the set of each combination of parameters, take the mean euclidean distance across those sessions.
results = []

for combo in combinations:
    df_filt = df[
        (df['subject'] == combo[0]) & 
        (df['target'] == combo[1]) & 
        (df['clustMask'] == combo[2]) & 
        (df['sequence_type'] == combo[3])
    ]
    meanDist = round(pdist(df_filt[['X','Y','Z']], metric='euclidean').mean(),2)
    results.append(meanDist)

# Append this new column to the previously generated df.
combodf['mean_distance'] = results
combodf.head()

# Plotting

In [ ]:
groupVars = ['target', 'clustMask', 'sequence_type'] # Matching df column names
targetName = 'sgACC' # For plotting, what does the 'target' mask represent?
clusterRegion = 'dlPFC' # for plotting, what does the 'clusterMask' mask represent?
groupVarNames = [f'{targetName} mask', f'{clusterRegion} mask', 'Sequence Type'] # The names for each of the above groupVars for plotting purposes

# For nilearn brain views, to visualize the output coordinates of processing with the clustMask type mentioned below, must match exactly.
colorDict = {
    'orig': 'red',
    'dilate_5': 'green',
    'erode_1': 'blue'
}


In [ ]:
def plot_significance_brackets(ax, comparisons, y_min, y_max):
    """
    Plot significance brackets and stars on an axis.
    
    Parameters:
    -----------
    ax : matplotlib axis
        The axis to plot on
    comparisons : list of dict
        List of comparisons, each containing 'pos1', 'pos2', and 'pval'
    y_min : float
        Minimum y-value in the data
    y_max : float
        Maximum y-value in the data
    
    Returns:
    --------
    tuple : (new_y_min, new_y_max) adjusted limits to accommodate brackets
    """
    y_range = y_max - y_min

    if not comparisons:
        return (y_min, y_max)

    bracket_height_start = y_max + 0.02 * y_range
    bracket_spacing = 0.08 * y_range

    for comp_idx, comp in enumerate(comparisons):
        x1 = comp['pos1']
        x2 = comp['pos2']
        pval = comp['pval']

        # Determine star based on p-value
        if pval < 0.001:
            star = '***'
        elif pval < 0.01:
            star = '**'
        elif pval < 0.05:
            star = '*'
        else:
            star = 'ns'

        # Calculate bracket height (stack them)
        bracket_y = bracket_height_start + (comp_idx * bracket_spacing)
        star_y = bracket_y + 0.002 * y_range

        # Draw horizontal line
        ax.plot([x1, x2], [bracket_y, bracket_y], 'k-', linewidth=1.5)

        # Draw vertical ticks on both ends
        tick_height = 0.015 * y_range
        ax.plot([x1, x1], [bracket_y - tick_height, bracket_y], 'k-', linewidth=1.5)
        ax.plot([x2, x2], [bracket_y - tick_height, bracket_y], 'k-', linewidth=1.5)

        # Add star
        ax.text((x1 + x2) / 2, star_y, star, ha='center', va='bottom',
                fontsize=14, fontweight='bold')

    # Return adjusted y-limits to accommodate all brackets
    max_bracket_y = bracket_height_start + (len(comparisons) - 1) * bracket_spacing
    return (y_min - 0.05 * y_range, max_bracket_y + 0.10 * y_range)

## Nilearn based plotting and visuals

### Single Subject plot

In [ ]:
if plot_nilearn:
    df_subj = df[(df['subject'] == 3)].copy()
    df_subj['colors'] = df_subj['clustMask'].map(colorDict)

    plot_coords = np.array(df_subj[['X','Y','Z']])
    plot_colors = list(df_subj['colors'])
    view = plotting.view_markers(plot_coords, plot_colors, marker_size=5)

    # Viewing options
    # view.open_in_browser()
    view

### Plot the mean coordinates for each subject

In [ ]:
if plot_nilearn:
    df_subj_centroid = df.groupby('subject')[['X', 'Y', 'Z']].mean()
    df_subj['colors'] = df_subj['clustMask'].map(colorDict)

    plot_coords = np.array(df_subj_centroid[['X','Y','Z']])
    # plot_colors = list(df_subj['colors'])
    view = plotting.view_markers(plot_coords, 'red', marker_size=5)
    # view.open_in_browser()
    view

## Raincloud Plot

In [ ]:
targetName = 'sgACC' # For plotting, what does the 'target' mask represent?
clusterRegion = 'dlPFC' # for plotting, what does the 'clusterMask' mask represent?

# Data column names and associated plotting names (2 variables, items must match)
groupVars = ['sequence_type', 'target', 'clustMask'] # Matching df column names
groupVarNames = ['Sequence Type', f'{targetName} mask', f'{clusterRegion} mask']

# Sequence of statistical tests and plotting
orderVec = [['se_e2', 'se', 'me'],
            ['seed', 'network'],
            ['erode_1', 'orig', 'dilate_5']]

# Display names for third plot
nameVec = [['Single Echo', 'Multi-Echo', 'Multi-Echo + ICA'],
           ['Seed', 'Network'],  
           ['Eroded', 'Original', 'Dilated']]  

### Statistical tests

In [ ]:
# Storage for statistical results
stat_results = {}

for idx, var in enumerate(groupVars):
    # Aggregate across other within-subject factors so each subject
    # has one value per level of the current factor
    agg_df = combodf.groupby(['subject', var])['mean_distance'].mean().reset_index()
    agg_df = agg_df.sort_values('subject')
    agg_df[var] = pd.Categorical(agg_df[var], categories=orderVec[idx], ordered=True)

    # Get groups in the correct order
    groups = orderVec[idx]
    group_data = [agg_df[agg_df[var] == val]['mean_distance'].values for val in groups]

    # Perform statistical test (paired t-test for 2 groups, RM-ANOVA for more)
    if len(groups) == 2:
        # Calculate and report the difference between the two groups
        mean_diff = group_data[0].mean() - group_data[1].mean()
        print(f'Difference between {groups[0]} and {groups[1]}: {mean_diff}')

        # Perform paired t-test
        stat, p_val = ttest_rel(group_data[0], group_data[1])
        stat_name = 't'

        print(f'Between {groups} {stat_name} test')
        print(f'Stat {stat}')
        print(f'P-val {p_val}')

        # Store single comparison result
        stat_results[var] = {
            'test': 'ttest',
            'stat': stat,
            'p_val': p_val,
            'stat_name': stat_name,
            'groups': groups,
            'comparisons': [
                {
                    'group1': groups[0],
                    'group2': groups[1],
                    'pval': p_val,
                    'pos1': 0,
                    'pos2': 1
                }
            ]
        }
    else:
        # Repeated measures ANOVA
        aovrm = AnovaRM(data=agg_df, depvar='mean_distance', subject='subject', within=[var])
        res = aovrm.fit()

        stat = res.anova_table['F Value'].values[0]
        p_val = res.anova_table['Pr > F'].values[0]
        stat_name = 'F'

        print(f'Among {groups} Repeated Measures ANOVA')
        print(res)

        # Post-hoc: pairwise paired t-tests with Bonferroni correction
        n_comparisons = len(groups) * (len(groups) - 1) // 2

        print(f'\n{groupVarNames[idx]} - Post-hoc Paired t-tests (Bonferroni corrected):')
        print('=' * 70)

        pairwise_results = []
        for g1, g2 in itertools.combinations(range(len(groups)), 2):
            t_stat, t_pval = ttest_rel(group_data[g1], group_data[g2])
            corrected_pval = min(t_pval * n_comparisons, 1.0)
            result = {
                'group1': groups[g1],
                'group2': groups[g2],
                'meandiff': group_data[g2].mean() - group_data[g1].mean(),
                't_stat': t_stat,
                'p_raw': t_pval,
                'p_corrected': corrected_pval,
                'reject': corrected_pval < 0.05
            }
            pairwise_results.append(result)
            sig = '*' if result['reject'] else ''
            print(f"  {result['group1']:>8} vs {result['group2']:<8}: "
                  f"diff={result['meandiff']:>7.3f}, t={result['t_stat']:>7.3f}, "
                  f"p_raw={result['p_raw']:.4f}, p_corr={result['p_corrected']:.4f} {sig}")
        print()

        # Extract significant comparisons for plotting
        group_positions = {group: i for i, group in enumerate(groups)}
        comparisons = []
        for r in pairwise_results:
            if r['reject']:
                comparisons.append({
                    'group1': r['group1'],
                    'group2': r['group2'],
                    'pval': r['p_corrected'],
                    'pos1': group_positions[r['group1']],
                    'pos2': group_positions[r['group2']]
                })

        # Sort comparisons by distance between groups (shortest first)
        comparisons.sort(key=lambda x: abs(x['pos2'] - x['pos1']))

        # Store results
        stat_results[var] = {
            'test': 'anova',
            'stat': stat,
            'p_val': p_val,
            'stat_name': stat_name,
            'groups': groups,
            'comparisons': comparisons
        }

print('\n' + '='*60)
print('Statistical analysis complete. Results stored for plotting.')
print('='*60)

### Raincloud plot - half violin, box, and rainplot.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

boxplot_kwargs = {'linewidth': 0.5}

for idx, (var, varPlot) in enumerate(zip(groupVars, groupVarNames)):
    # Convert the column to categorical with specified order
    combodf[var] = pd.Categorical(combodf[var], categories=orderVec[idx], ordered=True)

    testVar = pt.RainCloud(data=combodf, x=var, y='mean_distance', ax=axes[idx], 
        palette='Set2', hue=var, bw=0.3, width_viol=0.7, width_box=0.3,
        orient='v', alpha=0.6, dodge=False, pointplot=False, point_size=3, 
        linewidth=2, jitter=0.15, box_showfliers=False, box_linewidth=2)
    
    axes[idx].set_xticklabels(nameVec[idx])
    
    # Open up the plot to ensure the peaks for each distribution are visible.
    percSpread = 0.4 # Defines how much you want to extend past current limits.
    xmin, xmax = axes[idx].get_xlim()
    axes[idx].set_xlim((xmin-(percSpread), xmax+(percSpread)))

    # Get statistical results from the analysis cell
    results = stat_results[var]
    stat = results['stat']
    p_val = results['p_val']
    stat_name = results['stat_name']
    comparisons = results['comparisons']
    
    # Get y-axis bounds
    y_max = combodf['mean_distance'].max()
    y_min = combodf['mean_distance'].min()
    
    # Plot significance brackets and get adjusted y-limits
    new_y_min, new_y_max = plot_significance_brackets(
        ax=axes[idx],
        comparisons=comparisons,
        y_min=y_min,
        y_max=y_max
    )
    
    # Apply adjusted y-limits
    axes[idx].set_ylim(new_y_min, new_y_max)
    
    # Add title with statistics (two lines, different sizes)
    axes[idx].set_title(f'{varPlot}', fontsize=14, fontweight='bold', pad=22)
    axes[idx].text(0.5, 1.01, f'{stat_name}={stat:.2f}, p={p_val:.4f}',
                   transform=axes[idx].transAxes, ha='center', va='bottom',
                   fontsize=12)
    axes[idx].set_xlabel('', fontsize=11)
    axes[idx].set_ylabel('Mean Pairwise Distance (mm)', fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'Rainplot.svg'))
plt.show()